# Cálculo estructural 3D — sin IA, sin clave de API (Colab)
Ejecuta las celdas **en orden** (▶). Entrega un **paquete de revisión (ZIP)** que puedes descargar y compartir para que lo revisen.

> ⚠️ Ayuda al cálculo. Requiere revisión y firma de un ingeniero responsable. Calcula esfuerzos y desplazamientos; **no verifica resistencia según norma**.

## 1. Descargar el código e instalar dependencias

In [ ]:
import os, subprocess
from google.colab import userdata

REPO, RAMA = 'mrubilar84-ship-it/cloud', 'claude/hola-d69076'
try:
    token = userdata.get('GITHUB_TOKEN')    # solo si el repositorio es privado
except Exception:
    token = None

if not os.path.isdir('/content/cloud'):
    url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
    r = subprocess.run(['git', 'clone', '-b', RAMA, url, '/content/cloud'], capture_output=True, text=True)
    if r.returncode:
        raise SystemExit('No se pudo clonar (¿repositorio privado sin GITHUB_TOKEN?):\n' + r.stderr.replace(token or '#', '***'))
    subprocess.run(['git', '-C', '/content/cloud', 'remote', 'set-url', 'origin', f'https://github.com/{REPO}.git'])

%cd /content/cloud
!pip install -q ifcopenshell ezdxf openseespy

## 2. Subir tus archivos
Selecciona **a la vez**: tu **DXF**, tu **JSON de secciones** y (opcional) tu **JSON de combinaciones** (formato en `ejemplos/combinaciones_demo.json`).

Para probar sin archivos propios, cambia `USAR_DEMO` a `True`.

In [ ]:
import json
from google.colab import files

USAR_DEMO = False

if USAR_DEMO:
    !python crear_dxf_edificio.py
    dxf, sec, comb = 'ejemplos/edificio_2pisos.dxf', 'ejemplos/secciones_demo.json', 'ejemplos/combinaciones_demo.json'
else:
    subidos = files.upload()
    dxf = next(n for n in subidos if n.lower().endswith('.dxf'))
    sec, comb = None, None
    for n in subidos:                       # el JSON se reconoce por su contenido, no por el nombre
        if n.lower().endswith('.json'):
            d = json.load(open(n))
            if 'secciones' in d: sec = n
            elif 'combinaciones' in d: comb = n
    if sec is None:
        raise SystemExit("Falta el JSON de secciones (debe tener las claves 'materiales' y 'secciones').")
print('DXF:', dxf, '| Secciones:', sec, '| Combinaciones:', comb or '(ninguna: se usará cada caso por separado)')

## 3. Calcular y generar el paquete de revisión

In [ ]:
from estructural.revision import calcular_paquete

PROYECTO = 'Mi proyecto'   # ← cambia el nombre
r = calcular_paquete(dxf, sec, comb, 'salida', PROYECTO)

print('Equilibrio global:', 'OK ✅' if r['equilibrio_ok'] else 'FALLA ❌ — NO usar estos resultados')
print('\nAdvertencias:')
for a in r['advertencias']:
    print(' -', a)

## 4. Ver la memoria de cálculo

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('salida/memoria_calculo.md').read()))

## 5. Descargar el paquete de revisión (ZIP)
Guárdalo; contiene la memoria, los resultados, el modelo calculado y tus archivos de entrada.

In [ ]:
files.download('salida/paquete_revision.zip')

## (Opcional) Guardar también en Google Drive
Descomenta y ejecuta para copiar el paquete a tu Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/calculo_estructural
# !cp salida/paquete_revision.zip /content/drive/MyDrive/calculo_estructural/